# Détection de Tweets Suspects — Analyse Exploratoire

Notebook d'exploration et de prétraitement (Partie 1). Le pipeline complet (prétraitement → vectorisation → entraînement → évaluation) est industrialisé avec **DVC** dans `src/` et `dvc.yaml`.

## 1. Chargement des données

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/raw/tweets_suspect.csv')
df.head()

In [ ]:
print('Dimensions :', df.shape)
print('Variables  :', list(df.columns))
df.info()

## 2. Valeurs manquantes et doublons

In [ ]:
print('Valeurs manquantes :\n', df.isnull().sum())
print('\nDoublons :', df.duplicated().sum())

## 3. Distribution des classes

Le jeu de données est **fortement déséquilibré** : la classe 1 (suspect) représente ~90 % des observations. Cela justifie l'usage de *class weights* / SMOTE et le suivi du F1-score, de l'AUC et du rappel de la classe minoritaire plutôt que de la seule *accuracy*.

In [ ]:
df['label'].value_counts(normalize=True).round(3)

In [ ]:
ax = df['label'].value_counts().sort_index().plot(kind='bar', color=['#4C72B0','#C44E52'])
ax.set_xticklabels(['Non suspect (0)','Suspect (1)'], rotation=0)
ax.set_title('Distribution des classes'); plt.show()

## 4. Longueur des tweets

In [ ]:
df['len'] = df['message'].str.len()
df['nwords'] = df['message'].str.split().apply(len)
df.groupby('label')[['len','nwords']].describe().T

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.histplot(data=df, x='len', hue='label', bins=40, ax=ax[0])
ax[0].set_title('Longueur (caractères)')
sns.boxplot(data=df, x='label', y='nwords', ax=ax[1])
ax[1].set_title('Nombre de mots'); plt.tight_layout(); plt.show()

## 5. Prétraitement du texte

Opérations (voir `src/preprocess.py`) : minuscules, suppression des URLs, des mentions `@user`, des hashtags, de la ponctuation/chiffres, des *stop words*, puis **stemming** (Porter).

In [ ]:
import sys; sys.path.append('../src')
from preprocess import clean_text
ex = df['message'].iloc[0]
print('AVANT :', ex)
print('APRÈS :', clean_text(ex))

In [ ]:
df['clean'] = df['message'].apply(clean_text)
df[['message','clean','label']].head()

## 6. Nuages de mots par classe

In [ ]:
from wordcloud import WordCloud
fig, ax = plt.subplots(1,2, figsize=(14,5))
for i,lab in enumerate([0,1]):
    txt = ' '.join(df[df.label==lab]['clean'])
    wc = WordCloud(width=600, height=350, background_color='white',
                   colormap='Blues' if lab==0 else 'Reds').generate(txt)
    ax[i].imshow(wc); ax[i].axis('off'); ax[i].set_title(f'Classe {lab}')
plt.tight_layout(); plt.show()

## 7. Suite du projet

La représentation TF-IDF, la comparaison des modèles (Logistic Regression, Naive Bayes, LinearSVC, Random Forest, XGBoost), la gestion du déséquilibre, le GridSearch et l'évaluation (matrice de confusion, ROC/AUC) sont exécutés via le pipeline DVC :

```bash
dvc repro        # reproduit tout le pipeline
dvc metrics show # affiche les métriques
```

Voir aussi `src/compare_models.py` et `src/optimize.py`, et le rapport PDF `reports/`.